# ChIP-seq Processing Pipeline

**Tier 3 — Applied Bioinformatics | Module 24 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 23 (TF Footprinting & ATAC-seq)*

---

**By the end of this notebook you will be able to:**
1. Describe the ChIP-seq experimental workflow and the role of Input/IgG controls
2. Run read trimming, alignment, and duplicate removal on ChIP-seq data
3. Call peaks with MACS3 for transcription factors and histone marks
4. Compute FRiP score and cross-correlation quality metrics
5. Visualize signal enrichment with deepTools heatmaps and profile plots

**Key resources:**
- [ENCODE ChIP-seq pipeline](https://www.encodeproject.org/chip-seq/)
- [Harvard HBC ChIP-seq training](https://hbctraining.github.io/Intro-ChIP-seq/)
- [MACS3 documentation](https://macs3-project.github.io/MACS/)
- [deepTools documentation](https://deeptools.readthedocs.io/)


---

[← Module Overview](../README.md) | [Next: Differential Binding & Annotation →](./02_differential_binding.ipynb)

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Optional dependencies — caught gracefully
try:
    import pyBigWig
    HAS_PYBIGWIG = True
except ImportError:
    HAS_PYBIGWIG = False

try:
    import matplotlib_venn as venn_mod
    HAS_VENN = True
except ImportError:
    HAS_VENN = False

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
print('Setup complete.')


## 1. ChIP-seq Experimental Design

Chromatin immunoprecipitation followed by sequencing (ChIP-seq) maps the genome-wide binding sites of transcription factors (TFs) or the locations of histone modifications. The core workflow involves crosslinking proteins to DNA, sonicating chromatin into short fragments, immunoprecipitating with an antibody specific to the protein or histone mark of interest, reversing crosslinks, and sequencing the recovered DNA. A matching **Input** (or IgG control) sample undergoes the same protocol minus the immunoprecipitation step, capturing background chromatin accessibility bias without any enrichment. The ratio of ChIP signal to Input signal forms the basis of all downstream peak calling.

| Experiment Type | Peak Shape | Recommended Reads | Example Marks |
|---|---|---|---|
| Transcription Factor (TF) | Narrow (< 500 bp) | 20–40 M | CTCF, GATA1, p53 |
| Active histone mark | Narrow | 30–50 M | H3K4me3, H3K9ac |
| Broad histone mark | Broad (> 5 kb) | 40–80 M | H3K27me3, H3K9me3 |
| Enhancer mark | Narrow/Mixed | 30–50 M | H3K27ac, H3K4me1 |

Antibody quality is the single most important variable in ChIP-seq. Only **ChIP-grade** antibodies (tested by the supplier for ChIP, ideally with a knockout validation) should be used; polyclonal antibodies validated for western blot alone are insufficient. For sequencing strategy, **paired-end** reads are strongly preferred over single-end because they allow accurate fragment-size estimation and more reliable removal of PCR duplicates (duplicate pairs must share both 5′ coordinates). ENCODE requires **≥ 2 biological replicates** for all ChIP-seq experiments to enable assessment of reproducibility via IDR (Irreproducibility Discovery Rate). Pooled pseudoreplicate analysis is also recommended to benchmark self-consistency.


## 2. Read QC and Trimming

**FastQC** generates per-read quality reports for each FASTQ file. The metrics most relevant to ChIP-seq are: (1) per-base quality scores — a drop below Q20 in the last 10 bp of 3′ ends is common and tolerable; (2) adapter content — high adapter content (> 20%) indicates short fragments and must be trimmed before alignment; (3) sequence duplication level — elevated duplication at the FastQC stage reflects library complexity, though most duplicates are removed by Picard later; (4) GC content distribution — an unexpected bimodal GC peak suggests contamination or strong PCR bias. **MultiQC** aggregates FastQC reports across all samples into a single interactive HTML report, making it easy to spot outliers.

**Trim Galore** wraps Cutadapt and automatically detects the most common Illumina adapter sequences (TruSeq, Nextera). For paired-end data use `--paired`; it will trim both mates and discard read pairs where either mate is shorter than the minimum length after trimming. The `--fastqc` flag reruns FastQC after trimming so both pre- and post-trim reports are available immediately. **fastp** is a faster C++-based alternative that handles adapter detection, quality filtering, and per-sample HTML reports in a single tool call; it is increasingly preferred for high-throughput pipelines.

Warning signs to act on before proceeding: adapter content > 20% means fragment sizes are very short (re-evaluate sonication); per-base quality dropping below Q20 at 3′ end beyond 10 bp may benefit from aggressive trimming (`--quality 25`); an unexpected secondary peak in GC content distribution often signals PCR amplification of a small subset of fragments (library complexity issue). If > 30% of read pairs are discarded after trimming, the library likely needs to be re-prepared.


In [ ]:
# Note: requires bioinformatics tools installed (conda environment)

# --- Step 1: Quality control with FastQC ---
# Run FastQC on raw FASTQ files (paired-end)
!fastqc sample_R1.fastq.gz sample_R2.fastq.gz -o qc/raw/ -t 4

# Aggregate reports with MultiQC
!multiqc qc/raw/ -o qc/raw/multiqc/

# --- Step 2: Adapter trimming with Trim Galore ---
# Automatically detects adapters; --fastqc re-runs QC after trimming
!trim_galore \
    --paired \
    --fastqc \
    --cores 4 \
    --quality 20 \
    --length 20 \
    --output_dir trimmed/ \
    sample_R1.fastq.gz sample_R2.fastq.gz

# Alternative: fastp (faster, similar features)
!fastp \
    -i sample_R1.fastq.gz -I sample_R2.fastq.gz \
    -o trimmed/sample_R1_trimmed.fastq.gz \
    -O trimmed/sample_R2_trimmed.fastq.gz \
    --detect_adapter_for_pe \
    --thread 4 \
    -h trimmed/fastp_report.html


## 3. Alignment with Bowtie2

**Bowtie2** is the standard short-read aligner for ChIP-seq. It builds an FM-index from the reference genome, enabling fast gapped alignment of short reads (typically 50–150 bp). For human experiments use the **hg38** reference assembly; for mouse use mm10 or mm39. After alignment, reads mapping to ENCODE blacklisted regions — repetitive elements, centromeres, and telomeres that produce artifactually high signal in nearly all ChIP-seq experiments — must be removed using `bedtools intersect -v`. The hg38 blacklist is available from the ENCODE portal.

A good ChIP-seq experiment should achieve **> 80% overall alignment rate** for trimmed paired-end reads against hg38. `samtools flagstat` reports total reads, mapped, properly paired, and duplicate counts at a glance. If alignment rate falls below 70%, first check the trimming log (adapter content), then verify the correct genome version was used. Low mapping can also reflect cross-species contamination or heavily degraded DNA.

Post-alignment filtering is critical for reducing noise: remove unmapped reads (`-F 4`), secondary alignments (`-F 256`), and low-confidence mappings with MAPQ < 30 (`-q 30`). Keep only properly paired reads (`-f 2`) to ensure both mates in a pair mapped concordantly. Combining these flags: `samtools view -F 4 -F 256 -q 30 -f 2 -b`. For paired-end data, `--no-mixed` and `--no-discordant` flags in Bowtie2 itself prevent mixed/discordant alignments from entering the BAM.


In [ ]:
# Note: requires bioinformatics tools installed (conda environment)

# --- Download or build Bowtie2 index (hg38) ---
# If not already indexed:
# !bowtie2-build hg38.fa hg38_index/hg38

# --- Align paired-end trimmed reads ---
!bowtie2 \
    -x /data/indices/hg38 \
    -1 trimmed/sample_R1_val_1.fq.gz \
    -2 trimmed/sample_R2_val_2.fq.gz \
    -p 8 \
    --no-mixed \
    --no-discordant \
    2> logs/sample_bowtie2.log | \
    samtools view -bS -F 4 -F 256 -q 30 -f 2 | \
    samtools sort -@ 4 -o aligned/sample.bam

# --- Index BAM ---
!samtools index aligned/sample.bam

# --- QC: Alignment statistics ---
!samtools flagstat aligned/sample.bam | tee logs/sample_flagstat.txt

# --- Remove blacklisted regions (ENCODE) ---
!bedtools intersect \
    -a aligned/sample.bam \
    -b hg38-blacklist.v2.bed \
    -v > aligned/sample_filtered.bam
!samtools index aligned/sample_filtered.bam


## 4. Duplicate Removal with Picard

Duplicate reads in ChIP-seq arise from two distinct sources. **PCR duplicates** are identical fragments amplified multiple times during library preparation; they reflect low library complexity and disproportionately inflate peak signal at highly enriched loci, creating false peaks. **Optical duplicates** are reads from physically proximate clusters on the flow cell that are mis-called as separate molecules; they are a technical artifact of the sequencer and are more prevalent on newer patterned flow cells (NovaSeq). In high-quality ChIP-seq experiments a total duplication rate above **50%** is a warning sign of library complexity problems; for TF ChIP the rate should ideally be < 30%.

**Picard MarkDuplicates** identifies duplicates by comparing the 5′ mapping coordinates of read pairs. Setting `REMOVE_DUPLICATES=true` physically removes duplicate reads from the output BAM rather than just flagging them. The `OPTICAL_DUPLICATE_PIXEL_DISTANCE` parameter should be set appropriately for the sequencer: **100** for HiSeq (non-patterned flow cell) and **2500** for NovaSeq (patterned flow cell). The output metrics file contains the `PERCENT_DUPLICATION` value and the estimated library size.

ENCODE uses two complementary complexity metrics computed from the duplication metrics file. The **PCR Bottleneck Coefficient (PBC1)** is the ratio of genomic positions with exactly one unique read to positions with at least one read; values > 0.7 are acceptable, > 0.9 are ideal. The **Non-Redundant Fraction (NRF)** is the number of distinct alignments divided by total alignments; NRF > 0.8 is acceptable, > 0.9 is ideal. After deduplication, re-run `samtools flagstat` to compare pre- and post-dedup read counts.


In [ ]:
# Note: requires bioinformatics tools installed (conda environment)

# --- Mark and remove PCR/optical duplicates with Picard ---
!picard MarkDuplicates \
    I=aligned/sample_filtered.bam \
    O=dedup/sample_dedup.bam \
    M=dedup/sample_dup_metrics.txt \
    REMOVE_DUPLICATES=true \
    OPTICAL_DUPLICATE_PIXEL_DISTANCE=2500 \
    VALIDATION_STRINGENCY=SILENT \
    TMP_DIR=./ \
    2> logs/picard_dedup.log

!samtools index dedup/sample_dedup.bam

# --- Compare flagstat before and after deduplication ---
!echo '=== BEFORE deduplication ===' && samtools flagstat aligned/sample_filtered.bam
!echo '=== AFTER deduplication ===' && samtools flagstat dedup/sample_dedup.bam


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Simulate duplication metrics (as would be output by Picard)
# In practice, parse dedup/sample_dup_metrics.txt
np.random.seed(42)

metrics_data = {
    'LIBRARY': ['ChIP_sample', 'Input_sample'],
    'UNPAIRED_READS_EXAMINED': [0, 0],
    'READ_PAIRS_EXAMINED': [35_420_000, 28_100_000],
    'UNMAPPED_READS': [1_200_000, 850_000],
    'UNPAIRED_READ_DUPLICATES': [0, 0],
    'READ_PAIR_DUPLICATES': [8_500_000, 3_200_000],
    'READ_PAIR_OPTICAL_DUPLICATES': [420_000, 180_000],
    'PERCENT_DUPLICATION': [0.240, 0.114],
    'ESTIMATED_LIBRARY_SIZE': [4_800_000, 11_200_000],
}
df_metrics = pd.DataFrame(metrics_data)

df_metrics['PCR_DUPS'] = (
    df_metrics['READ_PAIR_DUPLICATES'] - df_metrics['READ_PAIR_OPTICAL_DUPLICATES']
)
df_metrics['NRF'] = 1 - df_metrics['PERCENT_DUPLICATION']  # simplified approximation

print('Duplication Metrics Summary')
print('=' * 50)
cols = ['LIBRARY', 'READ_PAIRS_EXAMINED', 'PERCENT_DUPLICATION',
        'NRF', 'ESTIMATED_LIBRARY_SIZE']
print(df_metrics[cols].to_string(index=False))
print('\nENCODE thresholds: NRF > 0.8 (acceptable), > 0.9 (ideal)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
libraries = df_metrics['LIBRARY']
dup_pct = df_metrics['PERCENT_DUPLICATION'] * 100

axes[0].bar(libraries, dup_pct, color=['steelblue', 'coral'])
axes[0].axhline(20, color='red', linestyle='--', label='20% threshold')
axes[0].set_ylabel('Duplication Rate (%)')
axes[0].set_title('PCR Duplication Rate')
axes[0].legend()

# Stacked: optical vs PCR duplicates
optical = df_metrics['READ_PAIR_OPTICAL_DUPLICATES'] / 1e6
pcr = df_metrics['PCR_DUPS'] / 1e6
x = np.arange(len(libraries))
axes[1].bar(x, optical, label='Optical duplicates', color='tomato')
axes[1].bar(x, pcr, bottom=optical, label='PCR duplicates', color='steelblue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(libraries)
axes[1].set_ylabel('Read pairs (millions)')
axes[1].set_title('Duplicate Breakdown')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Peak Calling with MACS3

**MACS3** (Model-based Analysis of ChIP-seq 3) identifies genomic regions enriched in ChIP relative to Input using a local background model. For transcription factors it estimates the fragment length either from the read pair distance distribution (paired-end: `-f BAMPE`) or from the strand shift of forward vs reverse reads (single-end). The effective genome size (`-g hs` = 2.7 Gb for human) is used to compute the expected background Poisson rate. The local background is assessed at four scales: 1 kb, 10 kb, whole-genome, and Input coverage at the same position — the maximum of these is used as the expected value, making the model robust to variable chromatin accessibility.

The **q-value threshold** (`-q 0.05`) applies Benjamini–Hochberg FDR correction to the Poisson p-values computed at each candidate peak. Fold enrichment in the `.narrowPeak` file (column 7, `signalValue`) is the ChIP signal divided by the local Input background at each peak summit. The `peak` column (column 10) stores the offset in base pairs from the peak start to the summit position, enabling precise motif scanning. For broad histone marks such as H3K27me3 and H3K9me3, use `--broad --broad-cutoff 0.1`, which merges nearby enriched regions with a looser threshold into broad domains, without modelling a sharp summit.

The **narrowPeak format** (BED6+4) encodes: chromosome, start (0-based), end, name, score (0–1000 scaled), strand, signalValue (fold enrichment), pValue (−log₁₀), qValue (−log₁₀), and peak offset to summit. A **broadPeak** file adds a 7th extra column (signalValue) but omits the summit offset. After peak calling, always check the total peak count: TF ChIPs typically yield 5,000–50,000 narrow peaks; H3K4me3 typically yields 30,000–80,000 narrow peaks; H3K27me3 broad peaks may cover up to 30% of the genome.


In [ ]:
# Note: requires bioinformatics tools installed (conda environment)

# --- TF ChIP-seq: narrow peak calling ---
!macs3 callpeak \
    -t dedup/sample_dedup.bam \
    -c dedup/input_dedup.bam \
    -f BAMPE \
    -g hs \
    -n tf_sample \
    --outdir peaks/tf/ \
    -q 0.05 \
    --keep-dup all \
    2> logs/macs3_tf.log

# --- Histone mark: broad peak calling (H3K27ac as an example) ---
!macs3 callpeak \
    -t dedup/h3k27ac_dedup.bam \
    -c dedup/input_dedup.bam \
    -f BAMPE \
    -g hs \
    -n h3k27ac \
    --outdir peaks/histone/ \
    --broad \
    --broad-cutoff 0.1 \
    --keep-dup all \
    2> logs/macs3_h3k27ac.log

# Count called peaks
!echo 'TF peaks:'; wc -l peaks/tf/tf_sample_peaks.narrowPeak
!echo 'H3K27ac peaks:'; wc -l peaks/histone/h3k27ac_peaks.broadPeak


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Simulate a narrowPeak file (as output by MACS3)
# NarrowPeak columns: chr start end name score strand signalValue pValue qValue peak
np.random.seed(42)
n_peaks = 8500
chrom_choices = [f'chr{i}' for i in list(range(1, 23)) + ['X', 'Y']]
chrom_probs = np.array([248e6, 242e6, 198e6, 191e6, 181e6, 171e6, 159e6, 145e6,
                         138e6, 134e6, 135e6, 133e6, 114e6, 107e6, 102e6, 90e6,
                         83e6, 80e6, 58e6, 64e6, 47e6, 50e6, 155e6, 57e6])
chrom_probs = chrom_probs / chrom_probs.sum()

peaks_df = pd.DataFrame({
    'chrom': np.random.choice(chrom_choices, n_peaks, p=chrom_probs),
    'start': np.random.randint(1_000_000, 248_000_000, n_peaks),
    'name': [f'peak_{i}' for i in range(n_peaks)],
    'score': np.random.randint(100, 1000, n_peaks),
    'strand': '.',
    'signalValue': np.random.exponential(scale=5, size=n_peaks) + 1,
    'pValue': np.random.exponential(scale=10, size=n_peaks) + 2,
    'qValue': np.random.exponential(scale=5, size=n_peaks) + 1,
    'peak': np.random.randint(50, 300, n_peaks),  # offset to summit
})
peaks_df['end'] = peaks_df['start'] + peaks_df['peak'] * 2
peaks_df['width'] = peaks_df['end'] - peaks_df['start']

print(f'Total peaks called: {len(peaks_df):,}')
print(f'Median peak width: {peaks_df["width"].median():.0f} bp')
print(f'Mean fold enrichment: {peaks_df["signalValue"].mean():.2f}')
print('\nNarrowPeak format (first 3 rows):')
cols = ['chrom', 'start', 'end', 'name', 'score', 'strand',
        'signalValue', 'pValue', 'qValue', 'peak']
print(peaks_df[cols].head(3).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(peaks_df['width'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Peak Width (bp)')
axes[0].set_ylabel('Count')
axes[0].set_title('Peak Width Distribution')
axes[0].axvline(peaks_df['width'].median(), color='red', linestyle='--',
                label=f'Median: {peaks_df["width"].median():.0f} bp')
axes[0].legend()

axes[1].hist(peaks_df['signalValue'], bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Fold Enrichment')
axes[1].set_ylabel('Count')
axes[1].set_title('Peak Fold Enrichment Distribution')

peak_counts = peaks_df['chrom'].value_counts().reindex(chrom_choices, fill_value=0)
axes[2].bar(range(len(chrom_choices)), peak_counts.values,
            color='mediumpurple', edgecolor='white')
axes[2].set_xticks(range(0, len(chrom_choices), 4))
axes[2].set_xticklabels(chrom_choices[::4], rotation=45, ha='right', fontsize=8)
axes[2].set_xlabel('Chromosome')
axes[2].set_ylabel('Peak Count')
axes[2].set_title('Peaks per Chromosome')

plt.tight_layout()
plt.show()


## 6. Quality Metrics: FRiP Score

The **Fraction of Reads in Peaks (FRiP)** is defined as the number of mapped reads overlapping called peaks divided by the total number of mapped reads. It is the most widely used single-number summary of ChIP enrichment efficiency. A low FRiP score indicates poor antibody specificity, degraded chromatin (resulting in non-specific fragmentation), or insufficient sequencing depth relative to library complexity. FRiP integrates antibody quality, chromatin quality, and peak calling performance into one metric, making it an excellent early diagnostic.

ENCODE minimum acceptable thresholds are **≥ 5% FRiP for TFs and narrow histone marks** and **≥ 1% for broad marks** (because broad marks cover more of the genome, a lower percentage is expected). High-quality TF ChIP experiments from good antibodies (e.g., CTCF, H3K4me3) routinely achieve 10–30% FRiP. Input and IgG controls should have very low FRiP (< 1%) because they are not enriched for any specific locus. If a control has elevated FRiP it suggests the called peaks are largely background.

FRiP computation uses `featureCounts` or `bedtools intersect -c` to count how many reads overlap the peak regions, then divides by total mapped reads from `samtools flagstat`. A complementary quality plot is **deepTools `plotFingerprint`**: it ranks genomic bins by cumulative read count and plots the cumulative fraction of reads vs fraction of genome. An ideal ChIP-seq experiment produces a late-rising **J-curve** (most reads concentrated in few bins = peaks), while an Input rises as a diagonal (uniform coverage). A poor ChIP resembles the Input.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Simulate FRiP computation from peak overlaps
# In practice: frip = reads_in_peaks / total_mapped_reads
# reads_in_peaks from: bedtools intersect -a peaks.bed -b reads.bam -c
# or featureCounts -a peaks.narrowPeak -F SAF -o counts.txt sample.bam

np.random.seed(42)

samples = {
    'Sample': ['CTCF_rep1', 'CTCF_rep2', 'H3K27ac_rep1',
               'H3K27ac_rep2', 'Input_rep1', 'Input_rep2'],
    'Type': ['TF', 'TF', 'Histone', 'Histone', 'Input', 'Input'],
    'Total_Reads_M': [35.4, 32.1, 48.2, 51.3, 28.0, 30.5],
    'Mapped_Reads_M': [33.8, 30.5, 45.6, 48.7, 26.4, 28.9],
    'Peaks_Called': [12500, 11800, 42000, 45000, 0, 0],
    'Reads_in_Peaks_M': [8.4, 7.6, 22.3, 24.5, 0.18, 0.21],
}
df_qc = pd.DataFrame(samples)
df_qc['FRiP'] = df_qc['Reads_in_Peaks_M'] / df_qc['Mapped_Reads_M']
df_qc['Mapping_Rate'] = df_qc['Mapped_Reads_M'] / df_qc['Total_Reads_M']

print('ChIP-seq Quality Summary')
print('=' * 70)
cols_display = ['Sample', 'Type', 'Mapped_Reads_M', 'Peaks_Called',
                'FRiP', 'Mapping_Rate']
print(df_qc[cols_display].to_string(index=False, float_format='{:.3f}'.format))

print('\nENCODE FRiP thresholds: TF >= 0.05, Histone >= 0.01')
print('Flagged samples:')
for _, row in df_qc[df_qc['Type'] != 'Input'].iterrows():
    threshold = 0.05 if row['Type'] == 'TF' else 0.01
    status = 'PASS' if row['FRiP'] >= threshold else 'FAIL'
    print(f"  {row['Sample']}: FRiP = {row['FRiP']:.3f} ({status}, threshold = {threshold})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {'TF': 'steelblue', 'Histone': 'coral', 'Input': 'gray'}
bar_colors = [colors[t] for t in df_qc['Type']]
axes[0].bar(df_qc['Sample'], df_qc['FRiP'] * 100, color=bar_colors, edgecolor='white')
axes[0].axhline(5, color='steelblue', linestyle='--', alpha=0.7, label='TF threshold (5%)')
axes[0].axhline(1, color='coral', linestyle='--', alpha=0.7, label='Histone threshold (1%)')
axes[0].set_ylabel('FRiP (%)')
axes[0].set_title('Fraction of Reads in Peaks')
axes[0].set_xticklabels(df_qc['Sample'], rotation=45, ha='right')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in colors.items()]
axes[0].legend(handles=legend_elements + [
    plt.Line2D([0], [0], color='steelblue', linestyle='--', label='TF min (5%)'),
    plt.Line2D([0], [0], color='coral', linestyle='--', label='Histone min (1%)')
])

# Fingerprint plot simulation (cumulative ranked read distribution)
# Perfect Input: diagonal; Good ChIP: J-curve shape
rank_fracs = np.linspace(0, 1, 100)
input_curve = rank_fracs  # ideal input: linear (uniform coverage)
tf_curve = np.where(rank_fracs < 0.85, rank_fracs * 0.05, rank_fracs * 3.5 - 2.975)
tf_curve = np.clip(tf_curve, 0, 1)
axes[1].plot(rank_fracs, input_curve, 'gray',
             label='Input (linear = uniform coverage)', lw=2)
axes[1].plot(rank_fracs, tf_curve, 'steelblue',
             label='CTCF ChIP (J-curve = enrichment)', lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Diagonal (reference)')
axes[1].set_xlabel('Fraction of genome (ranked bins)')
axes[1].set_ylabel('Fraction of reads')
axes[1].set_title('Fingerprint Plot (plotFingerprint)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Visualization with deepTools

**BigWig files** are binary, indexed genome coverage tracks used for visualization in genome browsers such as IGV and the UCSC Genome Browser. `bamCoverage` converts a deduplicated BAM into a BigWig, normalizing read depth by **RPKM** (reads per kilobase per million mapped reads), CPM (counts per million), or BPM (bins per million). Use `--binSize 10` for transcription factors (high-resolution, narrow peaks) and `--binSize 50` for broad histone marks (lower resolution, large domains). The `--extendReads` flag extends each read to the estimated fragment length, filling the gap between paired-end reads and smoothing the coverage signal.

`computeMatrix` summarises read depth in windows around reference points or across gene body regions. In **reference-point** mode it centres each region at a fixed genomic coordinate (e.g., TSS) and extracts signal in flanking windows (`-b 3000 -a 3000` = ±3 kb). In **scale-regions** mode it scales each gene or region to the same length (e.g., gene body from TSS to TES), enabling comparison across genes of different lengths. Multiple BigWig files (ChIP, Input) can be provided simultaneously, producing a multi-sample matrix in one call.

`plotHeatmap` renders a heatmap where each row is one genomic region (gene, peak) and each column is a genomic bin; rows are sorted by total signal by default. `plotProfile` renders the mean signal ± SEM as a line plot. Adding `--kmeans N` to `plotHeatmap` clusters regions into N groups with distinct enrichment patterns (e.g., strong promoter TF binding vs. enhancer binding). A well-enriched TF ChIP shows a sharp peak centred at the summit or TSS; H3K4me3 shows a characteristic peak just downstream of the TSS; H3K36me3 shows gradual enrichment across gene bodies.


In [ ]:
# Note: requires bioinformatics tools installed (conda environment)

# --- Convert BAM to BigWig (RPKM normalized) ---
!bamCoverage \
    -b dedup/sample_dedup.bam \
    -o bigwig/sample.bw \
    --normalizeUsing RPKM \
    --binSize 10 \
    --numberOfProcessors 8 \
    --extendReads \
    2> logs/bamCoverage_sample.log

!bamCoverage \
    -b dedup/input_dedup.bam \
    -o bigwig/input.bw \
    --normalizeUsing RPKM \
    --binSize 10 \
    --numberOfProcessors 8 \
    --extendReads

# --- Compute matrix: signal +/-3 kb around TSS ---
!computeMatrix reference-point \
    -S bigwig/sample.bw bigwig/input.bw \
    -R genes_hg38.bed \
    --referencePoint TSS \
    -b 3000 -a 3000 \
    --binSize 10 \
    --numberOfProcessors 8 \
    -o matrix/tss_matrix.gz \
    --outFileSortedRegions matrix/tss_regions_sorted.bed \
    2> logs/computeMatrix.log

# --- Plot heatmap ---
!plotHeatmap \
    -m matrix/tss_matrix.gz \
    -out figures/heatmap_tss.png \
    --colorMap Blues \
    --whatToShow 'heatmap and colorbar' \
    --zMin 0 --zMax 10 \
    --samplesLabel 'ChIP' 'Input' \
    --regionsLabel 'Genes'

# --- Plot average profile ---
!plotProfile \
    -m matrix/tss_matrix.gz \
    -out figures/profile_tss.png \
    --samplesLabel 'ChIP' 'Input' \
    --plotTitle 'Signal around TSS (+/-3 kb)' \
    --perGroup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Simulate BigWig signal data (as would be read by pyBigWig)
# In practice: import pyBigWig; bw = pyBigWig.open('sample.bw')
# vals = bw.values('chr1', start, end, numpy=True)

np.random.seed(42)
n_bins = 600       # 600 bins x 10 bp = 6000 bp (+/-3 kb around TSS)
n_genes = 3000
bin_centers = np.linspace(-3000, 3000, n_bins)

def simulate_tss_signal(n_genes, n_bins, peak_height=8, peak_width=30):
    # Simulate ChIP-seq signal around TSS: sharp enrichment at centre
    center = n_bins // 2
    signal = np.zeros((n_genes, n_bins))
    for i in range(n_genes):
        base = np.random.exponential(0.3, n_bins)
        peak = peak_height * np.exp(
            -0.5 * ((np.arange(n_bins) - center) / peak_width) ** 2
        )
        gene_height = np.random.exponential(1.0)
        signal[i] = base + peak * gene_height
    return signal

chip_matrix = simulate_tss_signal(n_genes, n_bins, peak_height=8, peak_width=25)
input_matrix = np.random.exponential(0.4, (n_genes, n_bins))  # flat input

# Sort by total signal (as plotHeatmap --sortUsing sum)
sort_idx = chip_matrix.sum(axis=1).argsort()[::-1]
chip_sorted = chip_matrix[sort_idx]
input_sorted = input_matrix[sort_idx]

fig, axes = plt.subplots(1, 3, figsize=(14, 6))

# Heatmap - ChIP
im1 = axes[0].imshow(chip_sorted[:500], aspect='auto',
                      extent=[-3000, 3000, 0, 500],
                      vmin=0, vmax=12, cmap='Blues', origin='upper')
axes[0].set_xlabel('Distance from TSS (bp)')
axes[0].set_ylabel('Genes (sorted by signal)')
axes[0].set_title('ChIP Signal around TSS')
axes[0].axvline(0, color='red', linestyle='--', lw=0.8, alpha=0.7)
plt.colorbar(im1, ax=axes[0], label='RPKM')

# Heatmap - Input
im2 = axes[1].imshow(input_sorted[:500], aspect='auto',
                      extent=[-3000, 3000, 0, 500],
                      vmin=0, vmax=12, cmap='Greys', origin='upper')
axes[1].set_xlabel('Distance from TSS (bp)')
axes[1].set_title('Input Signal around TSS')
axes[1].axvline(0, color='red', linestyle='--', lw=0.8, alpha=0.7)
plt.colorbar(im2, ax=axes[1], label='RPKM')

# Average profile
axes[2].plot(bin_centers, chip_matrix.mean(axis=0), 'steelblue', lw=2, label='ChIP')
axes[2].fill_between(
    bin_centers,
    chip_matrix.mean(axis=0) - chip_matrix.std(axis=0)/np.sqrt(n_genes),
    chip_matrix.mean(axis=0) + chip_matrix.std(axis=0)/np.sqrt(n_genes),
    alpha=0.2, color='steelblue'
)
axes[2].plot(bin_centers, input_matrix.mean(axis=0), 'gray', lw=2, label='Input')
axes[2].axvline(0, color='red', linestyle='--', lw=1, label='TSS')
axes[2].set_xlabel('Distance from TSS (bp)')
axes[2].set_ylabel('Mean RPKM')
axes[2].set_title('Average Profile around TSS')
axes[2].legend()

plt.suptitle('deepTools TSS Analysis (simulated data)', y=1.02)
plt.tight_layout()
plt.show()

print(f'Peak signal at TSS (ChIP): '
      f'{chip_matrix[:, n_bins//2-5:n_bins//2+5].mean():.2f} RPKM')
print(f'Background signal (Input): {input_matrix.mean():.2f} RPKM')
print(f'Enrichment at TSS: '
      f'{chip_matrix[:, n_bins//2-5:n_bins//2+5].mean() / input_matrix.mean():.1f}x')


## Summary

This notebook has walked through the complete ChIP-seq processing pipeline from raw FASTQ files to quality-assessed, visualized signal tracks. Each step builds on the previous: poor quality reads produce poor alignments; elevated duplication reduces effective depth; liberal peak calling without Input normalization inflates false discoveries. Quality at every step compounds — a 20% adapter rate, 85% alignment, 25% duplication, and 10% FRiP together suggest a library that should be re-evaluated before expensive downstream analyses are undertaken.

| Pipeline Step | Tool | Key Output | Quality Threshold |
|---|---|---|---|
| Read QC | FastQC + MultiQC | HTML report | Q20+, < 20% adapters |
| Adapter trimming | Trim Galore / fastp | Trimmed FASTQ | > 20 bp reads |
| Alignment | Bowtie2 | BAM file | > 80% mapped |
| Alignment filtering | samtools | Filtered BAM | MAPQ ≥ 30 |
| Deduplication | Picard MarkDuplicates | Deduplicated BAM | NRF > 0.8 |
| Peak calling (TF) | MACS3 | narrowPeak | q < 0.05 |
| Peak calling (histone) | MACS3 --broad | broadPeak | q < 0.1 |
| Quality metrics | FRiP computation | QC table | TF ≥ 5%, Histone ≥ 1% |
| Track generation | bamCoverage (RPKM) | BigWig | — |
| Visualization | computeMatrix + plotHeatmap | PNG heatmap | — |

Proceed to **Notebook 2** for differential binding analysis between conditions (DiffBind), peak annotation to genomic features (ChIPseeker), GO enrichment, and integration with RNA-seq differentially expressed genes.


## 🧪 Exercises

1. **Experimental Design:** A collaborator wants to profile H3K27me3 (broad repressive mark) in two cell lines with 3 replicates each. What sequencing depth would you recommend per sample? Should they use paired-end or single-end sequencing? Justify your answer.

2. **FRiP Interpretation:** You run a CTCF ChIP-seq experiment and obtain a FRiP of 0.03 (3%). (a) Does this pass ENCODE thresholds? (b) List three experimental causes of low FRiP. (c) What pipeline steps would you investigate first?

3. **NarrowPeak Parsing:** Using the simulated `peaks_df` DataFrame created in Section 5, filter for peaks with fold enrichment > 10 and q-value > 3 (-log10). How many high-confidence peaks remain? Plot their genomic distribution.

4. **deepTools Matrix:** The heatmap in Section 7 shows ChIP signal around the TSS. How would you expect the pattern to differ for: (a) H3K36me3 (mark of active transcription elongation), (b) H3K27me3 (repressive mark)? Sketch or describe the expected profile.

5. **Pipeline Debugging:** A peer runs the pipeline and gets only 45% alignment rate and 8,000 peaks despite starting with 40M reads. List the diagnostic steps you would take and what tool output you would check first.


---

[← Module Overview](../README.md) | [Next: Differential Binding & Annotation →](./02_differential_binding.ipynb)

---
